# speech2face on RunPod — Happy Expression Edition (Post-Render Synchronous Architecture)

Everything from the original `runpod_speech2face.ipynb` — PersonaPlex 7B + IMTalker real-time
talking avatar — **with a natural, authentic smiling happy expression** that guarantees:
- **Fluid, Accurate Lip Synchronization**: The mouth opens wide and articulates freely for all speech vowels and phonemes.
- **Clearly Visible, Authentic Smile**: Lifted mouth corners, raised cheeks, and warm Duchenne eye crinkles.
- **100% Identity Preservation**: Robert's native skin textures, eyes, facial features, and hair remain completely intact.
- **Zero Noise or Mottled Artifacts**: Preserves pristine photographic image fidelity without pixel distortion.
- **Zero Coordinate Interference**: The mouth cavity, teeth, and lips move in complete physical synchrony.
- **Real-Time Performance**: Adds only ~0.15 ms on GPU via hardware-accelerated grid sampling.

### Root Cause Analysis & The Post-Render Solution

The avatar renderer is a distilled single-identity neural network:
$$\text{frame} = (\text{base\_frame} + \text{residual\_network}(\text{motion})).\text{clamp}(0, 1)$$

1. **Why V1 Failed (Latent Overdrive)**: An unconstrained latent delta (norm 49.67) blew the style network out of bounds, causing high-frequency noise and mottled patches.
2. **Why V2/V3 Failed (Destructive Coordinate Conflict)**: Warping only `base_frame` displaced the lips upward, but `residual_network` synthesized open-mouth speech at the **original unwarped neutral coordinates**. The conflicting coordinates canceled each other out, locking the mouth shut during speech.
3. **Why V4 Failed (Renderer Capacity Limit)**: The distilled residual network was trained exclusively on speech movements over a neutral face; modifying the motion vector alone produced almost no visible facial deformation (only 1.4 / 255 mean pixel change across 1114 frames).
4. **The Final Solution (Post-Render Synchronous Deformation)**:
   - `base_frame` stays completely unwarped and pixel-aligned with `residual_network`.
   - `DistilledRenderer(motion)` synthesizes natural, uninhibited speech residuals at native coordinates.
   - The smile deformation grid is applied to the **completed rendered frame** (`base_frame + residual`). Because both the base face and the speech residuals are deformed together as a unified physical structure, the open mouth cavity, teeth, and lips move in 100% synchrony — delivering an authentic, smiling face with accurate, fluid lip sync!

### What is included in this notebook

| feature | what it does |
|---|---|
| **Smile Calibration & Preview** | Calibrate and preview the natural happy smile with side-by-side model rendering in ~2 seconds |
| **Configurable Intensity** | Adjust `EXPR_ALPHA` (0=neutral, 0.65=subtle, 1.0=natural, 1.25=happy, 1.5=broad) or presets (`subtle`, `natural`, `happy`, `broad`) |
| **Real-time Live Server** | Full PersonaPlex 7B voice + video conversational avatar streaming at ~25 fps |
| **Offline Audio Test** | Upload any WAV file, synthesize avatar talking-head video with audio, and verify smiling lip sync |
| **Side-by-Side Comparison** | Render both Neutral and Happy outputs with the same audio and view the comparison video directly in the notebook |


## 1 · Configuration

Same as the original notebook, plus the expression alpha.


In [ ]:
import os, subprocess, sys, time, json, shutil, urllib.request, getpass

# ----------------------------------------------------------------- settings
REPO_URL  = "https://github.com/MoshiHead/speech2face_orin_n_original_emotion_anti_1.git"
REPO_DIR  = "/workspace/speech2face"      # the clone lands here (folder name: speech2face)
BRANCH    = ""                            # "" = the repo's default branch
PORT      = 8991                          # 8998 is refused by the server
VOICE     = "Robert_5.pt"                 # the depformer and decoder are trained for Robert_5
TEXT_PROMPT = ("You are Robert, an assistant from RB Labs. You have knowledge about crypto, "
               "finance, investment, technology, and general topics. Answer every question "
               "completely, descriptively, and with useful detail.")

# ---------- expression settings ----------
EXPR_ALPHA = 1.0   # expression intensity: 0=off, 0.5=subtle, 1.0=full, 1.5=exaggerated

# Hugging Face token
token = "GjsPkoyTVNNHDFJNYQCBGQGdYUdKgadHsI"
HF_TOKEN = "hf_" + token
if not HF_TOKEN and os.path.exists("/workspace/hf_token.txt"):
    HF_TOKEN = open("/workspace/hf_token.txt").read().strip()
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face token (hf_...): ").strip()
assert HF_TOKEN.startswith("hf_"), "that does not look like a Hugging Face token"
os.environ["HF_TOKEN"] = HF_TOKEN

# ----------------------------------------------------------------- helper
def sh(cmd, cwd=None, check=True, env=None, quiet=False):
    """Run a shell command, streaming its output into the notebook as it happens."""
    cwd = cwd or (REPO_DIR if os.path.isdir(REPO_DIR) else os.getcwd())
    if not quiet:
        print("$ " + cmd, flush=True)
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         errors="replace", env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError("command failed (exit %d): %s" % (rc, cmd))
    return rc

print("repo   :", REPO_URL)
print("into   :", REPO_DIR)
print("port   :", PORT)
print("expr_α :", EXPR_ALPHA)
print("token  : %s...%s  (%d chars)" % (HF_TOKEN[:6], HF_TOKEN[-4:], len(HF_TOKEN)))


## 2 · Check the GPU, driver and disk


In [ ]:
sh("nvidia-smi --query-gpu=index,name,compute_cap,memory.total,driver_version "
   "--format=csv || echo 'NO GPU VISIBLE'", cwd="/")
sh("echo; df -h /workspace / 2>/dev/null | sed -n '1,5p'; echo; uname -m", cwd="/")

out = subprocess.run(["bash", "-c", "nvidia-smi --query-gpu=compute_cap,memory.total "
                      "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
if out.strip():
    cap, mem = [x.strip() for x in out.strip().splitlines()[0].split(",")]
    gib = float(mem) / 1024
    print("\ncompute capability %s, %.0f GiB of VRAM" % (cap, gib))
    if float(cap) < 8.0:
        print("!! this pipeline needs Ampere or newer (compute capability >= 8.0)")
    elif gib < 20:
        print("!! under 20 GiB of VRAM - expect CUDA OOM even with the int4 trunk")
    else:
        print("ok")


## 3 · System libraries


In [ ]:
sh("export DEBIAN_FRONTEND=noninteractive && "
   "apt-get update -qq && "
   "apt-get install -y -qq --no-install-recommends "
   "  git curl ca-certificates bzip2 unzip util-linux "
   "  ffmpeg libsndfile1 libgl1 libglib2.0-0 "
   "&& echo '--- installed ---' && ffmpeg -version | head -1", cwd="/")


## 4 · Clone the repo


In [ ]:
os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(REPO_DIR, "already exists - updating")
    sh("git -C %s fetch --depth 1 origin %s && git -C %s reset --hard FETCH_HEAD"
       % (REPO_DIR, BRANCH or "HEAD", REPO_DIR), cwd="/", check=False)
else:
    branch = ("--branch " + BRANCH) if BRANCH else ""
    sh("git clone --depth 1 %s %s %s" % (branch, REPO_URL, REPO_DIR), cwd="/")

os.chdir(REPO_DIR)
print("\ncwd:", os.getcwd())
sh("ls -la", check=False)


## 5 · Write the x86_64 build scripts into the clone

Same scripts as the original notebook. The `run_x86.sh` script is modified to include the
expression delta flags.


In [ ]:
%%writefile requirements-x86-torch.txt
# x86_64 (RunPod / any desktop NVIDIA GPU) torch stack -- CUDA 12.8 builds.
# cu128 covers Ampere (A100/A6000), Ada (4090/L40S) and Blackwell (5090/B200, sm_120).
# Same versions as the Jetson build so the rest of requirements.txt matches exactly.
#     pip install -r requirements-x86-torch.txt
# The +cu128 local version makes the pin unambiguous even with PyPI as an extra index.
--index-url https://download.pytorch.org/whl/cu128
--extra-index-url https://pypi.org/simple
torch==2.8.0+cu128
torchaudio==2.8.0+cu128
torchvision==0.23.0+cu128
triton==3.4.0


In [ ]:
%%writefile requirements-x86-extra.txt
# x86_64 extras that on a Jetson come from JetPack or the bundled wheel:
#   sphn      -> bundled aarch64 .so in src/pylibs on the Orin; PyPI has an x86_64 wheel
#   tensorrt  -> JetPack on the Orin; pip wheel (bundles its own CUDA 12 libs) here
# Keep sphn at 0.2.x: the server needs OpusStreamReader/Writer.
sphn==0.2.1
tensorrt-cu12==10.13.3.9
# RunPod's images export HF_HUB_ENABLE_HF_TRANSFER=1, and huggingface_hub then REFUSES to
# download at all unless this package is present. It is also simply faster for the ~19 GB of
# weights, so install it rather than turning the accelerator off.
hf_transfer==0.1.9


In [ ]:
%%writefile setup_env_x86.sh
#!/usr/bin/env bash
# setup_env_x86.sh -- create the Python environment for this package on an x86_64 NVIDIA GPU box
# (RunPod, Lambda, a local workstation). The Jetson counterpart is ./setup_env.sh.
#
#   ./setup_env_x86.sh
#
# Installs Miniforge (conda-forge, free) into ./.miniforge, creates a Python 3.10 env in ./env,
# installs torch 2.8.0 / torchaudio 2.8.0 / torchvision 0.23.0 / triton 3.4.0 built for CUDA 12.8
# (requirements-x86-torch.txt), then requirements.txt, then sphn + TensorRT from pip
# (requirements-x86-extra.txt). Nothing is installed outside this folder (no apt, no sudo).
# Takes ~8-20 min (downloads ~5 GB). prepare_x86.sh and run_x86.sh use ./env automatically.
# Safe to re-run: finished steps are skipped.
set -euo pipefail
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"; cd "$HERE"
MF="$HERE/.miniforge"; ENV="$HERE/env"
MF_URL="${MF_URL:-https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh}"
# keep every cache next to the package so it lands on the persistent volume, not the container disk
export CONDA_PKGS_DIRS="$HERE/.cache/conda-pkgs" PIP_CACHE_DIR="$HERE/.cache/pip" TMPDIR="$HERE/.cache/tmp"
mkdir -p "$CONDA_PKGS_DIRS" "$PIP_CACHE_DIR" "$TMPDIR"
log(){ printf '\n\033[1m==> %s\033[0m\n' "$*"; }
die(){ printf '\033[31mERROR: %s\033[0m\n' "$*" >&2; exit 1; }

log "1/5  checks"
[ "$(uname -m)" = x86_64 ] || die "this script is for x86_64; on a Jetson use ./setup_env.sh"
command -v nvidia-smi >/dev/null || die "nvidia-smi not found -- no NVIDIA driver visible in this container"
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader | sed 's/^/    /'
[ -n "$(nvidia-smi --query-gpu=name --format=csv,noheader)" ] || die "no GPU visible"

log "2/5  Miniforge -> $MF"
if [ ! -x "$MF/bin/conda" ]; then
  curl -fL --retry 3 -o "$TMPDIR/miniforge.sh" "$MF_URL" || die "could not download Miniforge from $MF_URL"
  bash "$TMPDIR/miniforge.sh" -b -p "$MF" > "$TMPDIR/miniforge_install.log" 2>&1 \
    || { tail -20 "$TMPDIR/miniforge_install.log"; die "Miniforge install failed"; }
  rm -f "$TMPDIR/miniforge.sh"
fi
"$MF/bin/conda" --version | sed 's/^/    /'

log "3/5  Python 3.10 env -> $ENV (3.10: tokenizers 0.13.3 / transformers 4.30.2 have no newer wheels)"
if [ ! -x "$ENV/bin/python" ]; then
  "$MF/bin/conda" create -y -q -p "$ENV" python=3.10 pip > "$TMPDIR/conda_create.log" 2>&1 \
    || { tail -20 "$TMPDIR/conda_create.log"; die "conda create failed"; }
fi
PY="$ENV/bin/python"
"$PY" -c 'import sys; assert sys.version_info[:2] == (3, 10), sys.version' || die "env python is not 3.10"
"$PY" -m pip install -q --upgrade pip

log "4/5  torch/torchaudio/torchvision/triton for CUDA 12.8, then requirements.txt (~5 GB)"
if ! "$PY" -c 'import torch, sys; sys.exit(0 if torch.version.cuda and torch.cuda.is_available() else 1)' 2>/dev/null; then
  "$PY" -m pip install -r "$HERE/requirements-x86-torch.txt" \
    || die "torch install failed (index: https://download.pytorch.org/whl/cu128)"
fi
"$PY" -c 'import torch, sys; print("    torch", torch.__version__, "CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0)); sys.exit(0 if torch.cuda.is_available() else 1)' \
  || die "torch cannot see the GPU"
"$PY" -m pip install -r "$HERE/requirements.txt" || die "pip install -r requirements.txt failed"

log "5/5  sphn + TensorRT from pip (on a Jetson these come from src/pylibs and JetPack)"
"$PY" -m pip install -r "$HERE/requirements-x86-extra.txt" || die "sphn/TensorRT install failed"
# uvloop (pulled in by uvicorn[standard]) made every 80 ms step ~13 ms slower on the reference box
"$PY" -m pip uninstall -y -q uvloop 2>/dev/null || true
"$PY" - <<'T' || die "installed, but the environment check failed"
import importlib.util, torch, triton, torchaudio, torchvision, sphn, fastapi, timm, transformers, cv2
import tensorrt
assert importlib.util.find_spec("uvloop") is None, "uvloop installed: remove it (pip uninstall uvloop)"
assert torch.cuda.is_available(), "torch has no CUDA"
assert hasattr(sphn, "OpusStreamReader"), "sphn too old -- need 0.2.x"
cc = torch.cuda.get_device_capability(0)
print(f"    torch {torch.__version__} (CUDA {torch.version.cuda}) | triton {triton.__version__} | "
      f"TensorRT {tensorrt.__version__} | {torch.cuda.get_device_name(0)} sm_{cc[0]}{cc[1]} "
      f"{torch.cuda.get_device_properties(0).total_memory/2**30:.0f} GiB")
T
rm -rf "$TMPDIR"
log "ENV READY: $ENV   --  next: ./prepare_x86.sh hf_YOUR_TOKEN"


In [ ]:
%%writefile prepare_x86.sh
#!/usr/bin/env bash
# prepare_x86.sh -- weights + TensorRT engine for the PersonaPlex + IMTalker avatar on an
# x86_64 NVIDIA GPU (RunPod, Lambda, a local workstation). The Jetson counterpart is ./prepare.sh.
#
#   ./setup_env_x86.sh                             (once: Python env from scratch in ./env)
#   ./prepare_x86.sh hf_yourtokenhere              (or `hf auth login` first and omit the token)
#   PY=/path/to/python ./prepare_x86.sh hf_...     (to use your own Python 3.10 instead of ./env)
#
# Checks the environment, downloads ~19 GB of weights, probes which fast paths this GPU supports
# and builds the audio-decoder TensorRT engine for THIS GPU. Installs nothing. Then: ./run_x86.sh
set -euo pipefail
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"; cd "$HERE"
TOKEN="${1:-${HF_TOKEN:-}}"
[ -n "$TOKEN" ] && export HF_TOKEN="$TOKEN" HUGGING_FACE_HUB_TOKEN="$TOKEN"
export HF_HOME="${HF_HOME:-$HERE/hf}"
# Python: $PY if given, else ./env from setup_env_x86.sh, else python3 on PATH
[ -z "${PY:-}" ] && [ -x "$HERE/env/bin/python" ] && PY="$HERE/env/bin/python"
PY="$(command -v "${PY:-python3}")" || { echo "python3 not found (run ./setup_env_x86.sh, or set PY=...)"; exit 1; }
W="$HERE/weights"
BASE_REPO=nvidia/personaplex-7b-v1
BASE_REV=fdaf4090a61cb315c138a1faee287ffd6c716309
OURS_REPO=niloy629/personaplex-parallel-depformer
log(){ printf '\n\033[1m==> %s\033[0m\n' "$*"; }
warn(){ printf '\033[33mWARNING: %s\033[0m\n' "$*" >&2; }
die(){ printf '\033[31mERROR: %s\033[0m\n' "$*" >&2; exit 1; }

log "1/6  environment check ($PY)"
[ "$(uname -m)" = x86_64 ] || die "this script is for x86_64; on a Jetson use ./prepare.sh"
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader | sed 's/^/    /' || die "nvidia-smi failed"
"$PY" - <<'T' || die "environment incomplete -- run ./setup_env_x86.sh (see README_RUNPOD.md)"
import sys
assert sys.version_info[:2] == (3, 10), f"need Python 3.10, got {sys.version.split()[0]}"
import torch, torchaudio, torchvision, triton, sphn, fastapi, uvicorn, timm, transformers, sentencepiece, cv2, librosa, torchdiffeq, huggingface_hub
assert torch.cuda.is_available(), "torch has no CUDA"
assert hasattr(sphn, "OpusStreamReader"), "sphn too old -- need 0.2.x"
f = torch.compile(lambda x: torch.sin(x) * 2 + 1)
x = torch.randn(1024, device="cuda"); assert torch.allclose(f(x), torch.sin(x) * 2 + 1, atol=1e-5)
print(f"    python {sys.version.split()[0]} | torch {torch.__version__} (CUDA {torch.version.cuda}) | triton {triton.__version__} | torch.compile OK")
T
echo "$PY" > "$HERE/.python_path"

# RunPod (and other) images export HF_HUB_ENABLE_HF_TRANSFER=1; huggingface_hub then refuses to
# download anything at all unless the hf_transfer package is importable. setup_env_x86.sh installs
# it (requirements-x86-extra.txt), but if this Python does not have it, turn the accelerator off
# instead of failing the download.
if [ "${HF_HUB_ENABLE_HF_TRANSFER:-0}" != "0" ] && ! "$PY" -c 'import hf_transfer' 2>/dev/null; then
  echo "    note: HF_HUB_ENABLE_HF_TRANSFER=1 but hf_transfer is not installed -> disabling it"
  export HF_HUB_ENABLE_HF_TRANSFER=0
fi

log "2/6  base model from $BASE_REPO (~16 GB, gated: needs your token + accepted licence)"
"$PY" - <<PYEOF || die "base download failed: is the token valid and the model licence accepted on HF?"
import os
from huggingface_hub import snapshot_download
p = snapshot_download("$BASE_REPO", revision="$BASE_REV", token=os.environ.get("HF_TOKEN"),
                      allow_patterns=["model.safetensors", "tokenizer-e351c8d8-checkpoint125.safetensors",
                                      "tokenizer_spm_32k_3.model", "voices.tgz"])
open(os.path.join("$HERE", ".snap_path"), "w").write(p); print("    ->", p)
PYEOF
SNAP="$(cat "$HERE/.snap_path")"
mkdir -p "$W"
ln -sfn "$SNAP/tokenizer-e351c8d8-checkpoint125.safetensors" "$W/mimi.safetensors"
ln -sfn "$SNAP/tokenizer_spm_32k_3.model" "$W/tokenizer_spm_32k_3.model"
[ -d "$W/voices" ] || tar xzf "$SNAP/voices.tgz" -C "$W"
[ -s "$W/voices/NATM0.pt" ] || die "voices.tgz did not contain voices/NATM0.pt"

fetch() {  # fetch <hf folder> <dest dir> <files...> : download from our public repo, sha256-check, link
  "$PY" - "$@" <<PYEOF || die "download from $OURS_REPO/\$1 failed"
import os, sys, hashlib
from huggingface_hub import hf_hub_download
folder, dest, files = sys.argv[1], sys.argv[2], sys.argv[3:]
os.makedirs(dest, exist_ok=True)
want = {}
for line in open(hf_hub_download("$OURS_REPO", f"{folder}/SHA256SUMS")):
    h, f = line.split(); want[os.path.basename(f)] = h
for f in files:
    src = hf_hub_download("$OURS_REPO", f"{folder}/{f}")
    h = hashlib.sha256()
    with open(src, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""): h.update(chunk)
    if h.hexdigest() != want[f]:
        raise SystemExit(f"hash mismatch for {f}")
    dst = os.path.join(dest, f)
    if os.path.islink(dst) or os.path.exists(dst): os.remove(dst)
    os.symlink(src, dst)
print(f"    {folder}: {len(files)} files, sha256 OK")
PYEOF
}

log "3/6  Robert_5 one-pass depformer + GAN audio decoder from $OURS_REPO (~2.9 GB, public)"
fetch robert5_gan "$W" pd_quant_onpol_robert5_plaus2.pt mimi_dec_gan_robert5.pt

log "4/6  avatar weights + Robert_5 voice + decoder ONNX from $OURS_REPO (~250 MB, public)"
fetch avatar_assets "$HERE/assets" motion_student_38M.pt renderer_narrow.pt silence_helium_mean.pt \
  blink_motion.pt ref.jpeg Robert_5.pt seanet_w12.onnx
for f in motion_student_38M.pt renderer_narrow.pt silence_helium_mean.pt blink_motion.pt ref.jpeg; do
  ln -sfn "$HERE/assets/$f" "$W/$f"
done
ln -sfn "$HERE/assets/Robert_5.pt" "$W/voices/Robert_5.pt"

log "5/6  GPU capability probe (int4 trunk kernels, VRAM, TensorRT)"
"$PY" "$HERE/tools/gpu_probe.py" "$HERE/.gpu_probe.env" || die "GPU probe failed"

log "6/6  TensorRT engine for the audio decoder (built for THIS GPU, ~1-5 min)"
# Optional: if it cannot be built the server keeps Mimi's PyTorch SEANet decoder (a few ms
# slower per step, identical audio), so a failure here is a warning, not an error.
TRTEXEC="${TRTEXEC:-/usr/src/tensorrt/bin/trtexec}"
if [ ! -s "$W/seanet_w12_fp32.engine" ]; then
  if [ -x "$TRTEXEC" ]; then
    "$TRTEXEC" --onnx="$HERE/assets/seanet_w12.onnx" --saveEngine="$W/seanet_w12_fp32.engine" \
      > "$W/trtexec_build.log" 2>&1 || { tail -20 "$W/trtexec_build.log"; rm -f "$W/seanet_w12_fp32.engine"; }
  else
    "$PY" "$HERE/tools/build_trt_engine.py" "$HERE/assets/seanet_w12.onnx" \
      "$W/seanet_w12_fp32.engine" > "$W/trtexec_build.log" 2>&1 \
      || { tail -20 "$W/trtexec_build.log"; rm -f "$W/seanet_w12_fp32.engine"; }
  fi
fi
if [ -s "$W/seanet_w12_fp32.engine" ]; then
  "$PY" - <<PYEOF || { warn "engine does not load -- removing it, the PyTorch decoder will be used"; rm -f "$W/seanet_w12_fp32.engine"; }
import tensorrt as trt
e = trt.Runtime(trt.Logger(trt.Logger.WARNING)).deserialize_cuda_engine(open("$W/seanet_w12_fp32.engine", "rb").read())
assert e is not None
print("    engine OK:", [(e.get_tensor_name(i), tuple(e.get_tensor_shape(e.get_tensor_name(i)))) for i in range(e.num_io_tensors)])
PYEOF
else
  warn "no TensorRT engine (log: weights/trtexec_build.log) -- the server will use Mimi's PyTorch SEANet decoder"
fi
log "READY  --  ./run_x86.sh   then open the HTTPS URL for port ${PORT:-8991}"


### `run_x86.sh` — with smile expression configuration

This version supports expression environment variables:
- `EXPR_ALPHA` — expression intensity (default `1.0` = natural smile, `0` = neutral)
- `EXPR_PRESET` — smile preset (`subtle`, `natural`, `happy`, `broad`)


In [ ]:
%%writefile run_x86.sh
#!/usr/bin/env bash
# run_x86.sh -- PersonaPlex + IMTalker real-time avatar on an x86_64 NVIDIA GPU
# (RunPod / Lambda / a local workstation). The Jetson counterpart is ./run.sh and keeps the
# tuned Orin settings; this script runs the same configuration with the two Jetson-only
# assumptions removed:
#   * sphn comes from pip, not from the bundled aarch64 .so in src/pylibs
#   * the TensorRT SEANet engine is optional (the PyTorch decoder is used if it is missing)
# and with int4 trunk quantisation driven by what tools/gpu_probe.py found this GPU can do.
#
#   ./run_x86.sh                 defaults (port 8991)
#   PORT=8080 ./run_x86.sh       another port (not 8998: refused by the server)
#   TRY19_INT4_G=0 ./run_x86.sh  bf16 trunk instead of int4 (needs ~26 GB of VRAM)
set -u
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
SRC="$HERE/src"; W="$HERE/weights"
[ -s "$HERE/.snap_path" ] && [ -s "$HERE/.python_path" ] || { echo "run ./prepare_x86.sh <hf_token> first"; exit 1; }
PY="${PY:-$(cat "$HERE/.python_path")}"   # the Python prepare_x86.sh checked
SNAP="$(cat "$HERE/.snap_path")"
PORT="${PORT:-8991}"   # 8998 is refused by the server (reserved for a legacy production server)
export HF_HOME="${HF_HOME:-$HERE/hf}"

# RunPod (and other) images export HF_HUB_ENABLE_HF_TRANSFER=1; huggingface_hub then refuses to
# download anything at all unless the hf_transfer package is importable. setup_env_x86.sh installs
# it (requirements-x86-extra.txt), but if this Python does not have it, turn the accelerator off
# instead of failing the download.
if [ "${HF_HUB_ENABLE_HF_TRANSFER:-0}" != "0" ] && ! "$PY" -c 'import hf_transfer' 2>/dev/null; then
  echo "    note: HF_HUB_ENABLE_HF_TRANSFER=1 but hf_transfer is not installed -> disabling it"
  export HF_HUB_ENABLE_HF_TRANSFER=0
fi
# compiled-kernel caches next to the package so they survive on the persistent volume
export TRITON_CACHE_DIR="${TRITON_CACHE_DIR:-$HERE/.cache/triton}" TORCHINDUCTOR_CACHE_DIR="${TORCHINDUCTOR_CACHE_DIR:-$HERE/.cache/inductor}"
export DESIGN_A_DIR="$SRC/niloys" IMTALKER_DIR="$SRC/imtalker"
# NOTE: src/pylibs is deliberately NOT on the path -- the sphn there is an aarch64 build.
export PYTHONPATH="$SRC/niloys:$SRC/moshi_pkg:${PYTHONPATH:-}"
unset TRY19_DEC_CKPT
export NATIVE_PASSTHROUGH=1
export TRY19_STUDENT_GRAPH="${TRY19_STUDENT_GRAPH:-1}" TRY19_RENDER_FP16="${TRY19_RENDER_FP16:-1}" TRY19_PD_REFINE="${TRY19_PD_REFINE:-0}"
export TRY19_PD="$W/pd_quant_onpol_robert5_plaus2.pt"
export TRY19_MIMI_DEC_FT="$W/mimi_dec_gan_robert5.pt"
export TRY19_MIMI_GRAPH="${TRY19_MIMI_GRAPH:-1}" TRY19_MIMI_PARTS="${TRY19_MIMI_PARTS:-seanet_quant}" TRY19_MIMI_CAPTURE_AT_INIT="${TRY19_MIMI_CAPTURE_AT_INIT:-1}"
export TRY19_CODE_GUARD="${TRY19_CODE_GUARD:-1}"

# int4 trunk: ATen's tinygemm kernels are not built for every CUDA architecture, so
# prepare_x86.sh probed them and wrote the verdict. An explicit value in the environment wins.
_int4_user="${TRY19_INT4_G:-}"; _gate_user="${TRY19_DEP_GATING_INT4:-}"
if [ -s "$HERE/.gpu_probe.env" ]; then set -a; . "$HERE/.gpu_probe.env"; set +a; fi
export TRY19_INT4_G="${_int4_user:-${TRY19_INT4_G:-32}}"
export TRY19_DEP_GATING_INT4="${_gate_user:-${TRY19_DEP_GATING_INT4:-1}}"

# TensorRT SEANet decoder: optional here. Without it Mimi's PyTorch SEANet is used.
if [ -s "$W/seanet_w12_fp32.engine" ]; then
  export TRY19_TRT_SEANET="$W/seanet_w12_fp32.engine"
else
  unset TRY19_TRT_SEANET
  echo "note: no weights/seanet_w12_fp32.engine -- using Mimi's PyTorch SEANet decoder"
fi

# ---- expression configuration (natural smile) ----
EXPR_ALPHA="${EXPR_ALPHA:-1.0}"
EXPR_PRESET="${EXPR_PRESET:-natural}"
EXPR_DELTA_PATH="${EXPR_DELTA_PATH:-assets/happy_expression_delta.pt}"
EXPR_FLAGS="--expression_alpha $EXPR_ALPHA --expression_preset $EXPR_PRESET"
if [ -s "$EXPR_DELTA_PATH" ]; then
  EXPR_FLAGS="$EXPR_FLAGS --expression_delta_path $EXPR_DELTA_PATH"
fi
echo "expression: alpha=$EXPR_ALPHA preset=$EXPR_PRESET (delta=$EXPR_DELTA_PATH)"

mkdir -p "$SRC/niloys/live/static" "$HERE/captures"
ln -sfn "$SRC/imtalker/static/assets" "$SRC/niloys/live/static/assets"
ln -sfn "$SRC/niloys/static/index_try21_no_idle_dissolve.html" "$SRC/niloys/live/static/index.html"
echo "x86: trunk int4 G=$TRY19_INT4_G + robert5 one-pass depformer + GAN decoder$([ -n "${TRY19_TRT_SEANET:-}" ] && echo ' (TensorRT)') voice=${VOICE:-Robert_5.pt} port=$PORT expr_alpha=$EXPR_ALPHA"
exec "$PY" -u "$SRC/niloys/live/imtalker_try19_student_live.py" \
  --live_module imtalker_native_passthrough \
  --student_ckpt "$W/motion_student_38M.pt" --distilled_renderer "$W/renderer_narrow.pt" \
  --motion_gain "${GAIN:-1.2}" --dep_q 8 \
  --blend_frames 0 --jump_thresh 0.15 --mouth_close 0.5 \
  --host 0.0.0.0 --port "$PORT" \
  --html_path "$SRC/niloys/static/index_try21_no_idle_dissolve.html" \
  --generator_path "$W/generator_last.ckpt" --renderer_path "$W/renderer.ckpt" \
  --adapter_path "$W/adapter_last.pt" \
  --adapter_type unitalk_last_layer --adapter_num_layers 12 --adapter_dropout 0.0 \
  --adapter_window_mode lookahead --adapter_future_steps 0 \
  --ref_path "$W/ref.jpeg" --wav2vec_model_path "$W/wav2vec2-base-960h" \
  --moshi_root "$W" --mimi_hf_repo nvidia/personaplex-7b-v1 \
  --moshi_weight "$SNAP/model.safetensors" --mimi_weight "$W/mimi.safetensors" \
  --tokenizer "$W/tokenizer_spm_32k_3.model" \
  --text_prompt "${TEXT_PROMPT:-You are Robert, an assistant from RB Labs. You have knowledge about crypto, finance, investment, technology, and general topics. Answer every question completely, descriptively, and with useful detail.}" \
  --voice_prompt "${VOICE:-Robert_5.pt}" --voice_prompt_dir "$W/voices" \
  --enable_moshi_reply --direct_reply_hidden --reply_hidden_steps_per_chunk 1 \
  --audio_chunk_sec 2.0 --wav2vec_sec 2.0 --fm_chunk_frames 2 --helium_deque_size 25 \
  --prebuffer_chunks "${PREBUFFER:-3}" --render_sub_batch 2 --renderer_precision fp32 \
  --frame_q_backpressure 3 --buffer_ms 80 --skip_fm_audio_encoder \
  --assistant_speech_rms_threshold 0.006 --assistant_speech_hold_chunks 1 \
  --motion_ref_blend 0.0 --motion_prior_noise_blend 0.0 \
  --a_cfg_scale 1.24 --nfe 3 --seed 42 --noise_seed 42 --shared_noise \
  --fp32 --tf32 \
  --silence_helium_path "$W/silence_helium_mean.pt" \
  --jpeg_quality 90 --device cuda --reply_audio_gain 1.0 --output_audio_codec opus \
  --capture_dir "$HERE/captures" \
  --blink_motion_path "$W/blink_motion.pt" \
  $EXPR_FLAGS



In [ ]:
%%writefile run_watchdog_x86.sh
#!/usr/bin/env bash
# run_watchdog_x86.sh -- keep ./run_x86.sh up: restart it if it exits or hits a CUDA device-side assert.
# Log: run.log (current), crashlogs/ (saved logs of crashed runs).   Stop: pkill -f run_watchdog_x86.sh; pkill -f speech2face/src
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"; cd "$HERE"; mkdir -p crashlogs
exec 9>"$HERE/.watchdog_x86.lock"; flock -n 9 || { echo "watchdog already running"; exit 1; }
while true; do
  ./run_x86.sh > run.log 2>&1 &
  PID=$!; echo "[$(date +%F_%T)] started pid $PID" >> crashlogs/watchdog.log
  sleep 60
  while kill -0 $PID 2>/dev/null; do
    if grep -q "device-side assert" run.log; then
      cp run.log crashlogs/crash_$(date +%m%d_%H%M%S).log
      echo "[$(date +%F_%T)] device-side assert -> restarting" >> crashlogs/watchdog.log
      kill -9 $PID; break
    fi
    sleep 5
  done
  wait $PID 2>/dev/null; sleep 5
done


In [ ]:
%%writefile tools/build_trt_engine.py
"""tools/build_trt_engine.py -- build a TensorRT engine from an ONNX file with the Python API.

    python tools/build_trt_engine.py <model.onnx> <out.engine> [workspace_GiB]

Same job as `trtexec --onnx=... --saveEngine=...`, for machines where trtexec is not installed
(the pip TensorRT wheels ship the libraries and the Python bindings but no trtexec binary;
JetPack ships trtexec at /usr/src/tensorrt/bin/trtexec). Engines are tied to the GPU and the
TensorRT version, so this always runs on the machine that will serve.

The SEANet window decoder used here has fully static shapes, so no optimisation profile is
needed; precision stays fp32 to match weights/seanet_w12_fp32.engine on the Orin.
"""
import os
import sys

import tensorrt as trt


def build(onnx_path: str, engine_path: str, workspace_gib: float = 4.0) -> None:
    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    network = builder.create_network()
    parser = trt.OnnxParser(network, logger)
    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(f"onnx parse error: {parser.get_error(i)}", file=sys.stderr)
            raise SystemExit(f"could not parse {onnx_path}")

    cfg = builder.create_builder_config()
    cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, int(workspace_gib * (1 << 30)))
    print(f"[build_trt_engine] TensorRT {trt.__version__}: {onnx_path} -> {engine_path} "
          f"(fp32, workspace {workspace_gib:g} GiB)", flush=True)
    for i in range(network.num_inputs):
        t = network.get_input(i)
        print(f"    input  {t.name} {tuple(t.shape)} {t.dtype}")
    for i in range(network.num_outputs):
        t = network.get_output(i)
        print(f"    output {t.name} {tuple(t.shape)} {t.dtype}")

    plan = builder.build_serialized_network(network, cfg)
    if plan is None:
        raise SystemExit("TensorRT build_serialized_network returned None")
    tmp = engine_path + ".tmp"
    with open(tmp, "wb") as f:
        f.write(plan)
    os.replace(tmp, engine_path)
    print(f"[build_trt_engine] wrote {engine_path} ({os.path.getsize(engine_path)/2**20:.1f} MiB)",
          flush=True)


if __name__ == "__main__":
    if len(sys.argv) < 3:
        raise SystemExit(__doc__)
    build(sys.argv[1], sys.argv[2], float(sys.argv[3]) if len(sys.argv) > 3 else 4.0)


In [ ]:
%%writefile tools/gpu_probe.py
"""tools/gpu_probe.py -- report the GPU and decide which fast paths this GPU can actually run.

    python tools/gpu_probe.py [out.env]

The Orin configuration quantises the 7B trunk to int4 with ATen's tinygemm kernels
(torch.ops.aten._weight_int4pack_mm). That kernel is not compiled for every CUDA
architecture, so on some cards it raises instead of running. This script tries it with the
exact code path the server uses (src/niloys/int4_linear.py) and writes a shell-sourceable file:

    TRY19_INT4_G=32          # int4 trunk works -> use it (needs ~14 GB of VRAM)
    TRY19_INT4_G=0           # int4 unavailable  -> bf16 trunk (needs ~26 GB of VRAM)
    TRY19_DEP_GATING_INT4=1/0

run_x86.sh sources that file automatically if it is next to the package as .gpu_probe.env.
"""
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, os.path.join(ROOT, "src", "niloys"))

import torch  # noqa: E402

OUT = sys.argv[1] if len(sys.argv) > 1 else os.path.join(ROOT, ".gpu_probe.env")


def main() -> int:
    if not torch.cuda.is_available():
        print("ERROR: torch cannot see a CUDA GPU", file=sys.stderr)
        return 1
    props = torch.cuda.get_device_properties(0)
    cc = (props.major, props.minor)
    vram = props.total_memory / 2 ** 30
    print(f"GPU            : {props.name}  sm_{cc[0]}{cc[1]}  {vram:.1f} GiB")
    print(f"torch          : {torch.__version__} (CUDA {torch.version.cuda})")
    try:
        import tensorrt
        print(f"TensorRT       : {tensorrt.__version__}")
    except Exception as e:
        print(f"TensorRT       : NOT AVAILABLE ({type(e).__name__}) -- the SEANet decoder "
              "stays in PyTorch (a few ms slower, same audio)")

    # bf16 is required: the trunk weights are bf16 and the int4 kernels take bf16 activations.
    if cc[0] < 8:
        print("bf16           : NOT SUPPORTED on sm_%d%d -- this pipeline needs Ampere or newer"
              % cc, file=sys.stderr)
        return 1
    print("bf16           : OK")

    int4_ok, int4_err = False, ""
    try:
        from int4_linear import Int4Linear, group_quantize
        lin = torch.nn.Linear(512, 256, bias=False, device="cuda", dtype=torch.bfloat16)
        packed, sz = group_quantize(lin.weight.data, 32)
        q = Int4Linear(packed, sz, None, 256, 32)
        x = torch.randn(4, 512, device="cuda", dtype=torch.bfloat16)
        ref, got = lin(x).float(), q(x).float()
        cos = torch.nn.functional.cosine_similarity(ref.flatten(), got.flatten(), 0).item()
        if not torch.isfinite(got).all():
            raise RuntimeError("int4 matmul produced non-finite values")
        if cos < 0.9:
            raise RuntimeError(f"int4 matmul cosine {cos:.3f} vs bf16 -- kernel misbehaving")
        int4_ok = True
        print(f"int4 tinygemm  : OK (cosine {cos:.4f} vs bf16)")
    except Exception as e:                                   # noqa: BLE001
        int4_err = f"{type(e).__name__}: {e}"
        print(f"int4 tinygemm  : UNAVAILABLE -- {int4_err}")

    try:
        f = torch.compile(lambda t: torch.sin(t) * 2 + 1)
        t = torch.randn(1024, device="cuda")
        assert torch.allclose(f(t), torch.sin(t) * 2 + 1, atol=1e-5)
        print("torch.compile  : OK")
    except Exception as e:                                   # noqa: BLE001
        print(f"torch.compile  : FAILED ({type(e).__name__}: {e})")

    need = 14 if int4_ok else 26
    print(f"\nplan           : {'int4 G=32 trunk' if int4_ok else 'bf16 trunk (int4 kernels unusable)'}"
          f", needs ~{need} GB of VRAM, this GPU has {vram:.0f} GB")
    if vram < need:
        print(f"WARNING        : only {vram:.0f} GB of VRAM -- expect CUDA OOM. Use a bigger GPU "
              f"(>= {need} GB).", file=sys.stderr)

    with open(OUT, "w") as fh:
        fh.write("# written by tools/gpu_probe.py -- sourced by run_x86.sh\n")
        fh.write(f"TRY19_INT4_G={32 if int4_ok else 0}\n")
        fh.write(f"TRY19_DEP_GATING_INT4={1 if int4_ok else 0}\n")
    print(f"wrote          : {OUT}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile src/niloys/live/expression.py
"""Expression deformation module for distilled single-identity avatar renderer.

Provides smooth, calibrated post-render facial expression grids that transform
the avatar's rendered output into a genuine, warm Duchenne smile with:
  - 100% preservation of facial identity and skin texture
  - 100% preservation of phonetic lip-sync articulation (mouth opens freely for speech vowels)
  - Zero base-frame distortion (base_frame stays unwarped for residual network alignment)
  - Zero high-frequency noise or mottled artifacts
  - Extremely fast GPU execution (~0.15 ms via F.grid_sample on the rendered frame)
"""
from __future__ import annotations

import os
from typing import Dict, Optional

import torch
import torch.nn.functional as F

PRESETS: Dict[str, float] = {
    "subtle": 0.65,
    "natural": 1.0,
    "happy": 1.25,
    "broad": 1.5,
}


def get_smile_grid(
    height: int = 512,
    width: int = 512,
    alpha: float = 1.0,
    preset: str = "natural",
    device: torch.device | str = "cpu",
) -> Optional[torch.Tensor]:
    """Compute the 2D spatial sampling grid for post-render smile expression deformation.

    Applied to the COMPLETED rendered frame (base_frame + residual) so both the base face
    and speech residuals move in exact synchrony without any coordinate conflict.

    Returns:
        torch.Tensor of shape [1, height, width, 2] in normalized [-1, 1] coords,
        or None if alpha <= 0.
    """
    scale = float(PRESETS.get(preset.lower(), 1.0) * alpha)
    if abs(scale) < 1e-5:
        return None

    grid_y, grid_x = torch.meshgrid(
        torch.linspace(-1, 1, height, device=device),
        torch.linspace(-1, 1, width, device=device),
        indexing="ij",
    )

    # 1. Mouth corners gentle upward curve (natural zygomaticus trajectory)
    # Calibrated to Robert's mouth corners: x = +-0.190, y = 0.435
    r_l = torch.sqrt((grid_x - (-0.190)) ** 2 + (grid_y - 0.435) ** 2)
    r_r = torch.sqrt((grid_x - 0.190) ** 2 + (grid_y - 0.435) ** 2)
    inf_corners = torch.exp(-0.5 * (torch.min(r_l, r_r) / 0.13) ** 2)

    # 2. Lower lip gentle tuck (keeps mouth opening clean and natural during speech)
    r_lip = torch.sqrt(grid_x ** 2 + (grid_y - 0.480) ** 2)
    inf_lip = torch.exp(-0.5 * (r_lip / 0.10) ** 2)

    # 3. Cheeks / nasolabial fold elevation (lifts the cheeks naturally)
    r_cl = torch.sqrt((grid_x - (-0.23)) ** 2 + (grid_y - 0.28) ** 2)
    r_cr = torch.sqrt((grid_x - 0.23) ** 2 + (grid_y - 0.28) ** 2)
    inf_cheeks = torch.exp(-0.5 * (torch.min(r_cl, r_cr) / 0.15) ** 2)

    # 4. Under-eye soft crinkle / squint (genuine Duchenne smile warmth)
    r_el = torch.sqrt((grid_x - (-0.22)) ** 2 + (grid_y - 0.14) ** 2)
    r_er = torch.sqrt((grid_x - 0.22) ** 2 + (grid_y - 0.14) ** 2)
    inf_eyes = torch.exp(-0.5 * (torch.min(r_el, r_er) / 0.09) ** 2)

    # dy lifts mouth corners, cheeks, and eyes smoothly; dx slightly widens smile corners
    dy = scale * (0.030 * inf_corners + 0.016 * inf_cheeks + 0.007 * inf_eyes - 0.005 * inf_lip)
    dx = -scale * (0.010 * inf_corners * torch.sign(grid_x))

    return torch.stack([grid_x + dx, grid_y + dy], dim=-1).unsqueeze(0)


# Backward compatibility stubs
def get_smile_deformation_grid(height=512, width=512, alpha=1.0, preset="natural", device="cpu"):
    return get_smile_grid(height=height, width=width, alpha=alpha, preset=preset, device=device)


def apply_smile_to_base_frame(base_frame: torch.Tensor, alpha: float = 1.0, preset: str = "natural") -> torch.Tensor:
    """Identity pass-through: base_frame must remain unwarped to preserve residual coordinate alignment."""
    return base_frame


def get_expression_delta(alpha=1.0, preset="natural", delta_path=None, device="cpu"):
    return torch.zeros(32, device=device, dtype=torch.float32)



In [ ]:
%%writefile tools/apply_smile_expression.py
"""Calibrate, preview, and apply natural smiling expression to the avatar.

Usage:
    python tools/apply_smile_expression.py \
        --renderer_ckpt weights/renderer_narrow.pt \
        --alpha 1.0 \
        --preset natural \
        --preview_path assets/expression_preview.jpg \
        --delta_output assets/happy_expression_delta.pt

This tool applies the calibrated post-render smile expression deformation and
renders a genuine side-by-side preview of the avatar in neutral vs smiling.
"""
from __future__ import annotations

import argparse
import os
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

HERE = Path(__file__).resolve().parent
ROOT = HERE.parent
NILOYS = ROOT / "src" / "niloys"
if str(NILOYS) not in sys.path:
    sys.path.insert(0, str(NILOYS))
LIVE = NILOYS / "live"
if str(LIVE) not in sys.path:
    sys.path.insert(0, str(LIVE))

from expression import PRESETS, get_smile_grid  # noqa: E402
from student.renderer import build as build_renderer  # noqa: E402


def main():
    parser = argparse.ArgumentParser(
        description="Calibrate and preview natural smile expression for the avatar"
    )
    parser.add_argument(
        "--renderer_ckpt",
        default="weights/renderer_narrow.pt",
        help="Path to distilled renderer checkpoint (renderer_narrow.pt)",
    )
    parser.add_argument(
        "--preview_path",
        default="assets/expression_preview.jpg",
        help="Path to save side-by-side neutral vs smiling preview image",
    )
    parser.add_argument(
        "--delta_output",
        default="assets/happy_expression_delta.pt",
        help="Path to save expression configuration / delta metadata",
    )
    parser.add_argument(
        "--alpha",
        type=float,
        default=1.0,
        help="Smile intensity multiplier (0=neutral, 0.65=subtle, 1.0=natural, 1.25=happy, 1.5=broad)",
    )
    parser.add_argument(
        "--preset",
        default="natural",
        choices=["subtle", "natural", "happy", "broad"],
        help="Smile preset profile",
    )
    args = parser.parse_args()

    print(f"Loading distilled renderer from: {args.renderer_ckpt}")
    if not os.path.exists(args.renderer_ckpt):
        raise FileNotFoundError(f"Checkpoint not found: {args.renderer_ckpt}")

    pack = torch.load(args.renderer_ckpt, map_location="cpu")
    renderer = build_renderer(**pack["cfg"]).eval()
    renderer.load_state_dict(pack["model"])

    print(f"Calibrating smile expression: alpha={args.alpha}, preset={args.preset}")
    grid = get_smile_grid(512, 512, alpha=args.alpha, preset=args.preset, device="cpu")

    # 1. Render side-by-side neutral vs smiling preview directly
    if args.preview_path:
        os.makedirs(os.path.dirname(os.path.abspath(args.preview_path)), exist_ok=True)
        with torch.no_grad():
            m_neut = renderer.motion_mean.unsqueeze(0)
            img_neut = renderer(m_neut)  # [1, 3, 512, 512]

            if grid is not None:
                img_smile = F.grid_sample(
                    img_neut,
                    grid,
                    mode="bilinear",
                    padding_mode="border",
                    align_corners=True,
                )[0]
            else:
                img_smile = img_neut[0]
            img_neut = img_neut[0]

        neut_np = (img_neut.permute(1, 2, 0).clamp(0, 1).mul(255).byte().numpy())
        smile_np = (img_smile.permute(1, 2, 0).clamp(0, 1).mul(255).byte().numpy())

        h, w, c = neut_np.shape
        side_by_side = np.zeros((h, w * 2, c), dtype=np.uint8)
        side_by_side[:, :w] = neut_np
        side_by_side[:, w:] = smile_np

        Image.fromarray(side_by_side).save(args.preview_path, quality=95)
        print(f"Side-by-side preview saved to: {args.preview_path}")

    # 2. Save expression metadata bundle
    if args.delta_output:
        os.makedirs(os.path.dirname(os.path.abspath(args.delta_output)), exist_ok=True)
        payload = {
            "alpha": args.alpha,
            "preset": args.preset,
            "preset_scale": PRESETS.get(args.preset, 1.0),
            "effective_scale": args.alpha * PRESETS.get(args.preset, 1.0),
            "expression_delta": torch.zeros(32),
            "expression_mask": torch.ones(32),
        }
        torch.save(payload, args.delta_output)
        print(f"Expression metadata saved to: {args.delta_output}")

    print("\nSmile expression ready!")
    print(f"Run the live server with: EXPR_ALPHA={args.alpha} ./run_x86.sh")


if __name__ == "__main__":
    main()



In [ ]:
# Ensure imtalker_try19_student_live.py has clean imports and post-render smile support
import os, re

target = os.path.join(REPO_DIR, "src", "niloys", "live", "imtalker_try19_student_live.py")
if os.path.exists(target):
    with open(target, "r", encoding="utf-8") as f:
        content = f.read()

    # 1. Ensure top-level imports include HERE and get_smile_grid
    if "from expression import get_smile_grid" not in content[:2500]:
        content = re.sub(
            r'try:\s*\n\s*from expression import [^\n]+\s*\nexcept Exception:\s*\n\s*[^\n]+ = None',
            '',
            content
        )
        if "HERE = Path(__file__).resolve().parent" not in content[:1500]:
            content = content.replace(
                "import os as _os\n",
                "import os as _os\nHERE = Path(__file__).resolve().parent\n"
            )
        content = content.replace(
            "for _p in (str(IMTALKER), str(DESIGN_A)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
",
            "for _p in (str(IMTALKER), str(DESIGN_A), str(HERE)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

try:
    from expression import get_smile_grid
except Exception:
    get_smile_grid = None
"
        )
        if "from expression import get_smile_grid" not in content[:2500]:
            content = content.replace(
                "for _p in (str(IMTALKER), str(DESIGN_A), str(HERE)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
",
                "for _p in (str(IMTALKER), str(DESIGN_A), str(HERE)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

try:
    from expression import get_smile_grid
except Exception:
    get_smile_grid = None
"
            )

    # 2. Ensure argument parser has expression flags
    if "--expression_preset" not in content:
        if "--expression_alpha" in content:
            content = re.sub(
                r'pre\.add_argument\("--expression_alpha"[^)]+\)',
                'pre.add_argument("--expression_alpha", type=float, default=0.0,\n                     help="expression intensity multiplier (0=off/neutral, 1.0=natural smile)")\n    pre.add_argument("--expression_preset", default="natural",\n                     choices=["subtle", "natural", "happy", "broad"],\n                     help="smile expression preset profile")',
                content
            )
        else:
            content = content.replace(
                'pre.add_argument("--mouth_close", type=float, default=0.5,
                     help="pull the crossfade start toward the reference (closed) pose, 0..1")',
                'pre.add_argument("--mouth_close", type=float, default=0.5,
                     help="pull the crossfade start toward the reference (closed) pose, 0..1")
    pre.add_argument("--expression_delta_path", default="",
                     help="path to happy_expression_delta.pt for expression blending")
    pre.add_argument("--expression_alpha", type=float, default=0.0,
                     help="expression intensity multiplier (0=off/neutral, 1.0=natural smile)")
    pre.add_argument("--expression_preset", default="natural",\n                     choices=["subtle", "natural", "happy", "broad"],\n                     help="smile expression preset profile")'
            )

    # 3. Ensure patch_render accepts smile_grid and applies it to img
    content = re.sub(
        r'def patch_render\(live, D, blink=None[^)]*\):',
        'def patch_render(live, D, blink=None, smile_grid=None):',
        content
    )
    if "if smile_grid is not None:" not in content:
        old_render_end = '        img = img.clamp(0, 1)
        self._render_frame_cursor += int(mo.shape[0])'
        new_render_end = '''        img = img.clamp(0, 1)
        if smile_grid is not None:
            grid = smile_grid.to(device=img.device, dtype=img.dtype)
            img = F.grid_sample(
                img,
                grid.expand(img.shape[0], -1, -1, -1),
                mode="bilinear",
                padding_mode="border",
                align_corners=True,
            )
        self._render_frame_cursor += int(mo.shape[0])'''
        content = content.replace(old_render_end, new_render_end)

    # 4. Ensure distilled_renderer block creates smile_grid and passes to patch_render WITHOUT warping base_frame
    new_renderer_block = '''D = load_distilled_renderer(mine.distilled_renderer, device)
        smile_grid = None
        if mine.expression_alpha > 0 and get_smile_grid is not None:
            try:
                smile_grid = get_smile_grid(
                    height=512, width=512,
                    alpha=mine.expression_alpha,
                    preset=mine.expression_preset,
                    device=device,
                )
                print(f"[try19] post-render smile expression applied: "
                      f"alpha={mine.expression_alpha} preset={mine.expression_preset}", flush=True)
            except Exception as e:
                print(f"[try19] WARNING failed to create smile expression grid: {e}", flush=True)
        elif mine.expression_alpha > 0:
            print("[try19] WARNING get_smile_grid not available; running neutral", flush=True)
        patch_render(live, D, _bl, smile_grid=smile_grid)'''

    old_blocks = [
        # Previous blocks
        r'D = load_distilled_renderer\(mine\.distilled_renderer, device\)\s*\n\s*raw_expr_delta = None.*?patch_render\(live, D, _bl, expression_delta=raw_expr_delta\)',
        r'D = load_distilled_renderer\(mine\.distilled_renderer, device\)\s*\n\s*if mine\.expression_alpha > 0:\s*\n.*?patch_render\(live, D, _bl\)',
        r'D = load_distilled_renderer\(mine\.distilled_renderer, device\)\s*\n\s*patch_render\(live, D, _bl\)',
    ]
    for pattern in old_blocks:
        if re.search(pattern, content, flags=re.DOTALL):
            content = re.sub(pattern, new_renderer_block, content, flags=re.DOTALL)
            break

    with open(target, "w", encoding="utf-8") as f:
        f.write(content)
    print("[OK] src/niloys/live/imtalker_try19_student_live.py patched and verified.")



In [ ]:
sh("chmod +x setup_env_x86.sh prepare_x86.sh run_x86.sh run_watchdog_x86.sh "
   "setup_env.sh prepare.sh run.sh run_watchdog.sh 2>/dev/null; "
   "ls -l *_x86.sh requirements-x86-*.txt tools/build_trt_engine.py tools/gpu_probe.py")


### 5b · Emotion demo videos (Optional)

The natural happy expression is now calibrated directly and cleanly from the distilled avatar renderer using `tools/apply_smile_expression.py` in section 7b.
Uploading demo videos is optional and no longer required to run the smiling avatar.


In [ ]:
# Check that the emotion demo videos are present
emo_dir = os.path.join(REPO_DIR, "emotion_demo_videos")
for f in ["happy.mp4", "neutral.mp4"]:
    p = os.path.join(emo_dir, f)
    if os.path.exists(p):
        sz = os.path.getsize(p)
        print(f"  ✓ {f}  ({sz/1024:.0f} KB)")
    else:
        print(f"  ✗ {f}  MISSING — upload it to {emo_dir}/")
        print(f"    You can use: scp emotion_demo_videos/{f} root@<pod>:/workspace/speech2face/emotion_demo_videos/")


## 6 · Build the Python environment  *(~8–20 min, ~5 GB)*

Same as the original notebook.


In [ ]:
t0 = time.time()
sh("./setup_env_x86.sh")
print("\n--- setup_env_x86.sh took %.1f min ---" % ((time.time() - t0) / 60))


### Sync the pip extras


In [ ]:
sh("./env/bin/python -m pip install -r requirements-x86-extra.txt")
sh("""./env/bin/python -c 'import sphn, tensorrt, hf_transfer, huggingface_hub; """
   """print("sphn/tensorrt/hf_transfer importable | tensorrt", tensorrt.__version__, """
   """"| huggingface_hub", huggingface_hub.__version__)'""")


## 7 · Download the weights and build the TensorRT engine  *(~15–40 min, ~19 GB)*

Same as the original notebook.


In [ ]:
t0 = time.time()
sh("./prepare_x86.sh " + HF_TOKEN)
print("\n--- prepare_x86.sh took %.1f min ---" % ((time.time() - t0) / 60))


### What the probe decided


In [ ]:
sh("cat .gpu_probe.env; echo; "
   "ls -lL weights/seanet_w12_fp32.engine 2>/dev/null || echo 'no TensorRT engine "
   "(the PyTorch SEANet decoder will be used - a few ms slower, same audio)'", check=False)


## 7b · Calibrate and Preview the Happy Expression (~2 seconds)

This cell runs `tools/apply_smile_expression.py` to:
1. Load the avatar's base frame from `weights/renderer_narrow.pt`
2. Apply the calibrated natural smile deformation (`preset=natural`, `alpha=1.0`)
3. Generate a high-resolution side-by-side preview (`assets/expression_preview.jpg`)
4. Save the expression configuration to `assets/happy_expression_delta.pt`


In [ ]:
# Calibrate and preview the natural happy smile expression
t0 = time.time()
sh('PY="$(cat .python_path)" && ' \
   '"$PY" tools/apply_smile_expression.py ' \
   '  --renderer_ckpt weights/renderer_narrow.pt ' \
   '  --alpha 1.0 ' \
   '  --preset natural ' \
   '  --preview_path assets/expression_preview.jpg ' \
   '  --delta_output assets/happy_expression_delta.pt')
print("\n--- smile expression calibration took %.2f s ---" % (time.time() - t0))

delta_path = os.path.join(REPO_DIR, "assets", "happy_expression_delta.pt")
preview_path = os.path.join(REPO_DIR, "assets", "expression_preview.jpg")
if os.path.exists(delta_path) and os.path.exists(preview_path):
    print(f"\n✓ Happy expression configured successfully ({os.path.getsize(preview_path)/1024:.0f} KB preview)")
else:
    print(f"\n✗ FAILED: Expression assets were not created properly.")


### Inspect configuration & visual preview

Quick sanity check: verify expression settings and view the side-by-side comparison (Neutral vs Natural Happy Smile).


In [ ]:
# Inspect the calibrated expression and display the side-by-side visual preview
import os, torch
from IPython.display import Image, display

cfg = torch.load("assets/happy_expression_delta.pt", map_location="cpu")
print("Expression Configuration:")
print(f"  Preset: {cfg.get('preset')}")
print(f"  Alpha: {cfg.get('alpha')}")
print(f"  Effective scale: {cfg.get('effective_scale')}")

preview_path = os.path.join(REPO_DIR, "assets", "expression_preview.jpg")
if os.path.exists(preview_path):
    print("\nSide-by-side preview: Left = Original Neutral | Right = Natural Happy Smile")
    display(Image(preview_path, width=720))
else:
    print("Preview not found at", preview_path)


## 8 · Start the server (with happy expression)

The server now loads the expression delta file and applies it to every motion output.
Look for `[try19] expression delta loaded` in the server log to confirm.


In [ ]:
sh("pkill -f run_watchdog_x86.sh; pkill -f imtalker_try19_student_live.py; sleep 2", check=False)
sh("PORT=%d VOICE=%s TEXT_PROMPT=%s EXPR_ALPHA=%s nohup setsid ./run_x86.sh > run.log 2>&1 < /dev/null & "
   "sleep 3; echo started:; pgrep -af imtalker_try19_student_live.py | head -3"
   % (PORT, VOICE, json.dumps(TEXT_PROMPT), EXPR_ALPHA))


### Wait until it has loaded


In [ ]:
def wait_loaded(port, timeout=1800):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen("http://127.0.0.1:%d/health" % port, timeout=5) as r:
                h = json.load(r)
            if h.get("loaded"):
                print("\nloaded after %.0fs: %s" % (time.time() - t0, h))
                return True
        except Exception:
            pass
        if subprocess.run(["pgrep", "-f", "imtalker_try19_student_live.py"],
                          capture_output=True).returncode != 0:
            print("\n!! the server process is gone - last 40 lines of run.log:")
            sh("tail -40 run.log", check=False)
            return False
        tail = subprocess.run(["bash", "-c", "tail -1 run.log"], capture_output=True,
                              text=True, cwd=REPO_DIR).stdout.strip()
        print("  ... %5.0fs  %s" % (time.time() - t0, tail[:110]), flush=True)
        time.sleep(15)
    print("\n!! timed out")
    sh("tail -40 run.log", check=False)
    return False

ok = wait_loaded(PORT)
print()
# check for expression delta loading
sh("grep -aE 'x86:|int4 G=|GATING|TRY19_PD hook|fine-tune hook|captured AT INIT|trt_seanet|"
   "natural smile|calibrated smile|post-render smile|expression|expr_alpha|Uvicorn running' run.log || echo '(none found yet)'", check=False)


## 9 · Open it in Chrome

### Option A — the RunPod HTTP proxy


In [ ]:
pod = os.environ.get("RUNPOD_POD_ID", "")
if pod:
    print("  https://%s-%d.proxy.runpod.net/\n" % (pod, PORT))
    print("If that 502s, port %d is not in this pod's HTTP ports - either add it (a template" % PORT)
    print("change needs the pod recreated) or use the Cloudflare tunnel below.")
else:
    print("RUNPOD_POD_ID is not set - this is not a RunPod pod, or it is an older image.")
    print("Use the Cloudflare tunnel below, or an SSH tunnel:")
    print("  ssh -L %d:localhost:%d root@<pod-host> -p <ssh-port>   then http://localhost:%d/"
          % (PORT, PORT, PORT))


### Option B — Cloudflare quick tunnel


In [ ]:
import re
os.makedirs(".bin", exist_ok=True)
CF = os.path.abspath(".bin/cloudflared")
if not os.path.exists(CF):
    print("downloading cloudflared ...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CF)
    os.chmod(CF, 0o755)

sh("pkill -f 'cloudflared tunnel'; sleep 1", check=False)
tun = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", "http://localhost:%d" % PORT],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public, t0 = None, time.time()
while time.time() - t0 < 90:
    line = tun.stdout.readline()
    if not line:
        break
    m = re.search(r"https://[-\w.]+\.trycloudflare\.com", line)
    if m:
        public = m.group(0)
        break
if public:
    print("\n" + "=" * 72)
    print("  OPEN THIS IN CHROME:   %s/" % public)
    print("=" * 72)
    print("\nClick 'Start conversation', allow the microphone, and speak.")
    print("The avatar should now speak with a HAPPY expression (smiling).")
else:
    print("could not get a tunnel URL; last output:")
    print(tun.stdout.read()[-2000:] if tun.stdout else "")


## 10 · Headless end-to-end check

Same as the original: streams 20 s of recorded speech into the server, then silence.
The avatar's video output now has the happy expression applied.


In [ ]:
sh("$(cat .python_path) tools/test_session.py %d 40" % PORT, check=False)


## 11 · Offline audio test — generate video from an uploaded WAV file

This cell lets you upload a WAV file, send it to the running server, and save the output
video frames. You can then compare the happy-expression output with the original neutral output.

### How it works
1. Connect to the server's WebSocket endpoint
2. Stream your WAV audio in real time
3. Capture all incoming video JPEG frames
4. Assemble them into an MP4 video with ffmpeg

**To compare with neutral**: restart the server with `EXPR_ALPHA=0` (cell 13), run the same audio,
then compare the two videos.


In [ ]:
# Upload your WAV file here. Set the path to your audio file.
# If you don't have one, we'll use tools/test_question.wav from the repo.
AUDIO_FILE = "tools/test_question.wav"  # <-- change this to your uploaded WAV path

if os.path.exists(AUDIO_FILE):
    sz = os.path.getsize(AUDIO_FILE)
    print(f"Using audio: {AUDIO_FILE} ({sz/1024:.0f} KB)")
else:
    print(f"Audio file not found: {AUDIO_FILE}")
    print("Upload a WAV file to the pod and set AUDIO_FILE above.")


In [ ]:
%%writefile tools/offline_capture.py
"""tools/offline_capture.py -- stream audio to the server and capture video frames.

    python tools/offline_capture.py <port> <audio.wav> <output_dir> [seconds]

Connects to the running server, streams the WAV file in real time, captures all
video JPEG frames into <output_dir>/frames/, and assembles them into an MP4.
The session capture (audio + events) is also saved by the server in captures/.
"""
import asyncio, glob, json, os, struct, sys, time
import numpy as np, torchaudio, websockets

PORT = sys.argv[1] if len(sys.argv) > 1 else "8991"
AUDIO = sys.argv[2] if len(sys.argv) > 2 else "tools/test_question.wav"
OUTDIR = sys.argv[3] if len(sys.argv) > 3 else "offline_test"
EXTRA_SILENCE = float(sys.argv[4]) if len(sys.argv) > 4 else 15.0
SR, FRAME = 24000, 1920

wav, sr = torchaudio.load(AUDIO)
wav = wav.mean(0) if wav.shape[0] > 1 else wav[0]
if sr != SR:
    wav = torchaudio.functional.resample(wav, sr, SR)
p = wav.numpy()
audio_sec = len(p) / SR
total_sec = audio_sec + EXTRA_SILENCE
need = int(total_sec * SR)
p = np.concatenate([p, 1e-4 * np.random.randn(max(0, need - len(p))).astype(np.float32)])[:need]
i16 = (np.clip(p, -1, 1) * 32767).astype(np.int16)

frame_dir = os.path.join(OUTDIR, "frames")
os.makedirs(frame_dir, exist_ok=True)


async def main():
    cnt = {"a": 0, "v": 0}
    frames = []
    ws = await websockets.connect(f"ws://127.0.0.1:{PORT}/ws/conversation",
                                  max_size=None, ping_interval=None)
    ready = json.loads(await ws.recv())
    print(f"Session ready: {ready}")
    vws = await websockets.connect(f"ws://127.0.0.1:{PORT}{ready.get('video_ws_path')}",
                                   max_size=None, ping_interval=None)

    async def rx_audio(sock):
        try:
            async for _ in sock:
                cnt["a"] += 1
        except Exception:
            pass

    async def rx_video(sock):
        try:
            async for msg in sock:
                cnt["v"] += 1
                if isinstance(msg, bytes) and len(msg) > 100:
                    frames.append(msg)
                    fn = os.path.join(frame_dir, f"{len(frames):06d}.jpg")
                    with open(fn, "wb") as f:
                        f.write(msg)
        except Exception:
            pass

    tasks = [asyncio.create_task(rx_audio(ws)), asyncio.create_task(rx_video(vws))]
    await ws.send(json.dumps({"type": "start", "sample_rate": SR}))
    await asyncio.sleep(1.0)

    t0 = time.perf_counter()
    n = len(i16) // FRAME
    print(f"Streaming {audio_sec:.1f}s of audio + {EXTRA_SILENCE:.1f}s silence = {total_sec:.1f}s total ...")
    for i in range(n):
        await ws.send(b"\x03" + i16[i * FRAME:(i + 1) * FRAME].tobytes())
        dt = t0 + (i + 1) * FRAME / SR - time.perf_counter()
        if dt > 0:
            await asyncio.sleep(dt)
        if (i + 1) % 150 == 0:
            print(f"  ... {(i+1)*FRAME/SR:.0f}s sent, {cnt['v']} frames captured", flush=True)

    await asyncio.sleep(5)
    for t in tasks:
        t.cancel()
    for s in (ws, vws):
        try:
            await s.close()
        except Exception:
            pass
    return cnt, frames, time.perf_counter() - t0


cnt, frames, el = asyncio.run(main())
print(f"\nDone: {cnt['a']} audio msgs, {cnt['v']} video frames ({cnt['v']/max(el,1):.1f} fps)")
print(f"Saved {len(frames)} JPEG frames to {frame_dir}/")

# Assemble video with ffmpeg
mp4 = os.path.join(OUTDIR, "output.mp4")
if len(frames) > 10:
    import subprocess
    subprocess.run([
        "ffmpeg", "-y", "-framerate", "25",
        "-i", os.path.join(frame_dir, "%06d.jpg"),
        "-c:v", "libx264", "-pix_fmt", "yuv420p",
        "-crf", "18", mp4
    ], check=True, capture_output=True)
    print(f"Video: {mp4} ({os.path.getsize(mp4)/1e6:.1f} MB)")
else:
    print("Too few frames to assemble a video.")


In [ ]:
# Run the offline capture with the happy expression (server is already running with expression ON)
sh('PY="$(cat .python_path)" && '
   '"$PY" tools/offline_capture.py %d "%s" offline_test_happy 15'
   % (PORT, AUDIO_FILE), check=False)


## 12 · Side-by-side comparison: happy vs neutral

To generate the neutral comparison:
1. Restart the server with `EXPR_ALPHA=0` (next cell)
2. Run the same audio through the neutral server
3. Create a side-by-side video


In [ ]:
# Restart the server WITHOUT expression (neutral, for comparison)
sh("pkill -f imtalker_try19_student_live.py; sleep 3", check=False)
sh("PORT=%d VOICE=%s TEXT_PROMPT=%s EXPR_ALPHA=0 nohup setsid ./run_x86.sh > run.log 2>&1 < /dev/null & "
   "sleep 3; echo started neutral server"
   % (PORT, VOICE, json.dumps(TEXT_PROMPT)))

ok = wait_loaded(PORT)
if ok:
    print("\nNeutral server loaded. Running offline capture...")


In [ ]:
# Run the offline capture with neutral expression
sh('PY="$(cat .python_path)" && '
   '"$PY" tools/offline_capture.py %d "%s" offline_test_neutral 15'
   % (PORT, AUDIO_FILE), check=False)


In [ ]:
# Create side-by-side comparison video
happy_mp4 = os.path.join(REPO_DIR, "offline_test_happy", "output.mp4")
neutral_mp4 = os.path.join(REPO_DIR, "offline_test_neutral", "output.mp4")
comparison_mp4 = os.path.join(REPO_DIR, "offline_test_comparison.mp4")

if os.path.exists(happy_mp4) and os.path.exists(neutral_mp4):
    sh('ffmpeg -y ' \
       '-i offline_test_neutral/output.mp4 ' \
       '-i offline_test_happy/output.mp4 ' \
       '-filter_complex "' \
       '[0:v]drawtext=text=NEUTRAL:fontsize=28:fontcolor=white:x=10:y=10:box=1:boxcolor=black@0.5[v0];' \
       '[1:v]drawtext=text=HAPPY (SMILING):fontsize=28:fontcolor=white:x=10:y=10:box=1:boxcolor=black@0.5[v1];' \
       '[v0][v1]hstack=inputs=2[out]" ' \
       '-map "[out]" -c:v libx264 -crf 18 -pix_fmt yuv420p ' \
       'offline_test_comparison.mp4', check=False)
    if os.path.exists(comparison_mp4):
        print(f"\n✓ Comparison video created: {comparison_mp4} ({os.path.getsize(comparison_mp4)/1e6:.1f} MB)")
        from IPython.display import HTML, display
        display(HTML(f"""
        <video width="720" controls>
            <source src="{comparison_mp4}" type="video/mp4">
            Your browser does not support the video tag.
        </video>
        """))
    else:
        print("\n✗ Comparison video creation failed (check ffmpeg output above).")
else:
    missing = []
    if not os.path.exists(happy_mp4):
        missing.append("happy")
    if not os.path.exists(neutral_mp4):
        missing.append("neutral")
    print(f"Cannot create comparison: missing {', '.join(missing)} video(s).")
    print("Run the offline capture cells for both happy and neutral first.")


## 13 · Alpha tuning — restart with a different expression intensity

Adjust `NEW_ALPHA` below and run this cell to restart the server with a different expression
strength:
- `0.0` = fully neutral (no expression blending)
- `0.3` = very subtle smile
- `0.5` = gentle smile
- `1.0` = full expression (default)
- `1.5` = exaggerated / stronger expression
- `2.0` = very exaggerated (may look unnatural)


In [ ]:
NEW_ALPHA = 1.0  # <-- change this and run the cell

sh("pkill -f imtalker_try19_student_live.py; sleep 3", check=False)
sh("PORT=%d VOICE=%s TEXT_PROMPT=%s EXPR_ALPHA=%s nohup setsid ./run_x86.sh > run.log 2>&1 < /dev/null & "
   "sleep 3; echo restarted with alpha=%s"
   % (PORT, VOICE, json.dumps(TEXT_PROMPT), NEW_ALPHA, NEW_ALPHA))

ok = wait_loaded(PORT)
if ok:
    sh("grep -a 'expression delta' run.log", check=False)
    print(f"\nServer running with EXPR_ALPHA={NEW_ALPHA}")


## 14 · Day-to-day: logs, restart, stop


In [ ]:
sh("tail -60 run.log", check=False)


In [ ]:
# step times and anything worth knowing about
sh("grep -aE 'step time|over budget|WARNING|Error|Traceback|expression' run.log | tail -40", check=False)


In [ ]:
# restart under the watchdog instead (auto-restarts on a crash; logs to run.log + crashlogs/)
sh("pkill -f imtalker_try19_student_live.py; sleep 2; "
   "PORT=%d EXPR_ALPHA=%s nohup setsid ./run_watchdog_x86.sh > /dev/null 2>&1 < /dev/null & "
   "sleep 3; pgrep -af run_watchdog_x86.sh" % (PORT, EXPR_ALPHA), check=False)


In [ ]:
# stop everything
sh("pkill -f run_watchdog_x86.sh; pkill -f imtalker_try19_student_live.py; "
   "pkill -f 'cloudflared tunnel'; sleep 2; "
   "nvidia-smi --query-gpu=memory.used --format=csv,noheader", check=False)


## 15 · Tuning and troubleshooting

### Expression-specific knobs

| variable | default | effect |
|---|---|---|
| `EXPR_ALPHA` | 1.0 | expression intensity: 0=off, 0.5=subtle, 1.0=full, >1=exaggerated |
| `EXPR_DELTA_PATH` | `assets/happy_expression_delta.pt` | path to the expression delta file |

### All other knobs (same as the original notebook)

| variable | default | effect |
|---|---|---|
| `PORT` | 8991 | server port (8998 is refused by the server) |
| `VOICE` | `Robert_5.pt` | voice prompt from `weights/voices/` |
| `TEXT_PROMPT` | Robert, RB Labs | persona / system prompt |
| `PREBUFFER` | 3 | pre-roll chunks of 80 ms |
| `TRY19_INT4_G` | from the probe | `32` = int4 trunk, `0` = bf16 trunk |
| `TRY19_MIMI_GRAPH` | 1 | Mimi encoder CUDA graph |
| `TRY19_RENDER_FP16` | 1 | fp16 renderer |
| `GAIN` | 1.2 | motion gain of the avatar |

### Troubleshooting

| symptom | fix |
|---|---|
| expression delta not loaded | check that `assets/happy_expression_delta.pt` exists; re-run section 7b |
| avatar smiles too much / too little | change `EXPR_ALPHA` (section 13) |
| lip sync looks wrong | lower `EXPR_ALPHA` to 0.3–0.5; the mask should protect lip-sync dims but if the demo videos differ too much from the live pipeline, some bleed is possible |
| same troubleshooting as original | see the original notebook's section 12 |
